# 9 — Authorization (IAM + CEL)

**AWS-side equivalent:** AgentCore Policy + Cedar
**GCP service:** IAM Conditions (CEL) + optional OPA sidecar for Cedar parity

AWS-side notebook: `notebooks/aws/09-agentcore-policy.ipynb`

_As of 2026-05: IAM Conditions uses a subset of CEL. Conditional bindings require IAM policy version 3 and an `etag` round-trip on every `setIamPolicy` call._


## Steps

1. Define role-keyed IAM bindings (`member` / `org-leader` / `hr-admin`) against the project
2. Add IAM Conditions (CEL expressions) for fine-grained gates (time-bounded, prefix-scoped)
3. **Grant Agent Engine runtime SA permissions** — BigQuery, Vertex AI, Cloud Trace
4. Sketch an OPA sidecar deployment for Cedar-equivalent policy parity


## Step 1: Load config


In [ ]:
# State plumbing — every notebook reads/writes config.json under the 'gcp' namespace.
import json
from pathlib import Path

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
print(f'project={PROJECT_ID} region={REGION}')


## Step 2: Define the OrgIQ role → GCP role mapping

AWS-side, OrgIQ uses three Cedar principals (`member`, `org-leader`, `hr-admin`). On GCP we map them to predefined IAM roles. `roles/aiplatform.user` lets the principal invoke Vertex AI; `roles/discoveryengine.viewer` lets them read Vertex AI Search; `roles/iam.securityReviewer` is the HR-admin escape hatch for read-everywhere audit. Replace the placeholder `group:` members with concrete Identity Platform / Workspace identities before applying.


In [ ]:
ROLE_BINDINGS = {
    'member': {
        'gcp_role': 'roles/aiplatform.user',
        'members': [f'group:orgiq-members@{PROJECT_ID}.iam.gserviceaccount.com'],
        'description': 'Default OrgIQ end-user; can call Vertex AI agents.',
    },
    'org-leader': {
        'gcp_role': 'roles/aiplatform.user',
        'members': [f'group:orgiq-leaders@{PROJECT_ID}.iam.gserviceaccount.com'],
        'description': 'Org-leader role; same Vertex AI access plus a higher rate budget.',
    },
    'hr-admin': {
        'gcp_role': 'roles/iam.securityReviewer',
        'members': [f'group:orgiq-hr@{PROJECT_ID}.iam.gserviceaccount.com'],
        'description': 'HR admin; read-only escape hatch across the project.',
    },
}

for role_name, spec in ROLE_BINDINGS.items():
    print(f'{role_name:12s} → {spec["gcp_role"]:38s} members={spec["members"][0]}')


## Step 3: Read the current project IAM policy (version 3)

Any operation that touches conditional bindings must request policy version 3. Always carry the returned `etag` through to the matching `set_iam_policy` call — omitting the `etag` lets a v1 client silently strip your conditions.


In [ ]:
from google.cloud import resourcemanager_v3
from google.iam.v1 import iam_policy_pb2, policy_pb2, options_pb2

projects_client = resourcemanager_v3.ProjectsClient()
PROJECT_RESOURCE = f'projects/{PROJECT_ID}'

policy = projects_client.get_iam_policy(
    request=iam_policy_pb2.GetIamPolicyRequest(
        resource=PROJECT_RESOURCE,
        options=options_pb2.GetPolicyOptions(requested_policy_version=3),
    )
)
print(f'policy.version={policy.version} etag={policy.etag.hex()[:16]}… bindings={len(policy.bindings)}')


## Step 4: Compose conditional bindings with CEL

Three CEL gates, mirroring the Cedar conditions in the AWS notebook:

| Role | Condition | CEL |
|------|-----------|-----|
| `member` | Until end-of-year sandbox | `request.time < timestamp('2026-12-31T00:00:00Z')` |
| `org-leader` | Business hours UTC | `request.time.getHours('UTC') >= 8 && request.time.getHours('UTC') <= 18` |
| `hr-admin` | Only `orgiq-*` resource name prefix | `resource.name.startsWith('projects/_/buckets/orgiq-')` |


In [ ]:
from google.type import expr_pb2

def _binding(role: str, members: list, title: str, expr: str, desc: str) -> policy_pb2.Binding:
    return policy_pb2.Binding(
        role=role,
        members=members,
        condition=expr_pb2.Expr(title=title, description=desc, expression=expr),
    )

new_bindings = [
    _binding(
        ROLE_BINDINGS['member']['gcp_role'],
        ROLE_BINDINGS['member']['members'],
        'orgiq-member-sandbox',
        "request.time < timestamp('2026-12-31T00:00:00Z')",
        'OrgIQ members get Vertex AI access through end of 2026.',
    ),
    _binding(
        ROLE_BINDINGS['org-leader']['gcp_role'],
        ROLE_BINDINGS['org-leader']['members'],
        'orgiq-leader-business-hours',
        "request.time.getHours('UTC') >= 8 && request.time.getHours('UTC') <= 18",
        'Org leaders may call Vertex AI 08:00–18:00 UTC.',
    ),
    _binding(
        ROLE_BINDINGS['hr-admin']['gcp_role'],
        ROLE_BINDINGS['hr-admin']['members'],
        'orgiq-hr-bucket-prefix',
        "resource.name.startsWith('projects/_/buckets/orgiq-')",
        'HR admins can audit only orgiq-prefixed buckets.',
    ),
]

for b in new_bindings:
    print(f'+ {b.role:38s} cond="{b.condition.title}"')


## Step 5: Stage the policy (dry-run, do NOT apply by default)

Mutating project IAM is destructive. The cell below assembles a v3 policy with the new bindings appended, prints what would change, and **only** calls `set_iam_policy` when `APPLY = True`. Flip it on once your group emails resolve to real principals.


In [ ]:
APPLY = False  # ← flip after replacing placeholder group: members

# Avoid double-adding identical bindings on a rerun: dedupe by (role, title, members).
existing_keys = {
    (b.role, b.condition.title if b.condition else '', tuple(b.members))
    for b in policy.bindings
}
to_add = [
    b for b in new_bindings
    if (b.role, b.condition.title, tuple(b.members)) not in existing_keys
]

staged = policy_pb2.Policy()
staged.CopyFrom(policy)
staged.version = 3
staged.bindings.extend(to_add)

print(f'Staged policy: {len(staged.bindings)} bindings (was {len(policy.bindings)}, +{len(to_add)} new)')

if APPLY:
    updated = projects_client.set_iam_policy(
        request=iam_policy_pb2.SetIamPolicyRequest(
            resource=PROJECT_RESOURCE,
            policy=staged,
        )
    )
    print(f'✓ Applied. New etag={updated.etag.hex()[:16]}…')
else:
    print('(dry-run — set APPLY = True to push these bindings)')


## Step 6: Persist the mapping to `config.json`

Downstream notebooks (10-identity-platform, 12-ag-ui-streaming) read this mapping to convert Identity Platform custom claims into GCP principals.


In [ ]:
gcp['role_bindings'] = ROLE_BINDINGS
gcp['iam_conditions'] = [
    {'title': b.condition.title, 'role': b.role, 'expression': b.condition.expression}
    for b in new_bindings
]
config_path.write_text(json.dumps(config, indent=2))
print('✓ config.json updated with gcp.role_bindings + gcp.iam_conditions')


## Step 7: Grant Agent Engine runtime service account permissions

Every Agent Engine (Reasoning Engine) container runs as
`service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com`.
That SA is NOT the notebook's user credential — it needs explicit grants for
every GCP service the deployed tools call at runtime.

| Notebook | Tools that need it | Role(s) |
|---|---|---|
| 08, 13 | `graph_query`, `find_experts`, `get_profile` | `bigquery.jobUser` + `bigquery.dataViewer` |
| 08 | `document_search` (RAG retrieval) | `aiplatform.user` |
| 17 | Cloud Trace export from spans | `cloudtrace.agent` |

Run this cell once per project. `add-iam-policy-binding` is idempotent — safe to re-run.


In [ ]:
import subprocess

# Derive the project number — it differs from the project ID.
_proj_number = subprocess.run(
    ['gcloud', 'projects', 'describe', PROJECT_ID, '--format=value(projectNumber)'],
    capture_output=True, text=True, check=True,
).stdout.strip()

AGENT_ENGINE_SA = f'service-{_proj_number}@gcp-sa-aiplatform-re.iam.gserviceaccount.com'
print(f'Agent Engine SA: {AGENT_ENGINE_SA}')

RUNTIME_GRANTS = [
    ('roles/bigquery.jobUser',    'create BQ query jobs from deployed tools'),
    ('roles/bigquery.dataViewer', 'read BQ table data from deployed tools'),
    ('roles/aiplatform.user',     'call Vertex AI APIs (RAG retrieval, embeddings) from deployed tools'),
    ('roles/cloudtrace.agent',    'export OpenTelemetry spans to Cloud Trace'),
]

member = f'serviceAccount:{AGENT_ENGINE_SA}'
for role, reason in RUNTIME_GRANTS:
    result = subprocess.run(
        [
            'gcloud', 'projects', 'add-iam-policy-binding', PROJECT_ID,
            f'--member={member}',
            f'--role={role}',
            '--condition=None',
            '--quiet',
        ],
        capture_output=True, text=True,
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {role:40s} ({reason})')
    if result.returncode != 0:
        print(f'    stderr: {result.stderr.strip()[:200]}')

print('\n✓ Agent Engine runtime SA grants applied (idempotent — safe to re-run)')


## Optional: OPA sidecar for Cedar-equivalent policy

IAM Conditions cover request-time gates, but they cannot express the entity-level relationships Cedar handles natively ("a user may invoke a tool iff that tool is in their team's allowed toolset"). To match the AWS notebook's expressiveness, the recommended pattern is an **OPA (Open Policy Agent) sidecar on Cloud Run** that the middleware queries before each agent invocation.

The cell below renders a Rego policy that mirrors the Cedar `permit (...) when { ... }` shape, plus the Cloud Run deploy command. We do not deploy it here — OPA is optional and adds another moving part. Treat this as a copy-paste starter.


In [ ]:
REGO_POLICY = '''
package orgiq.authz

default allow := false

# Members may invoke any tool tagged sensitivity <= medium.
allow if {
    input.principal.role == "member"
    input.action == "invoke"
    input.resource.sensitivity in {"low", "medium"}
}

# Org leaders may also invoke high-sensitivity tools.
allow if {
    input.principal.role == "org-leader"
    input.action == "invoke"
}

# HR admins may discover anything.
allow if {
    input.principal.role == "hr-admin"
    input.action == "discover"
}
'''

Path('opa_policy.rego').write_text(REGO_POLICY)
print('✓ wrote opa_policy.rego')
print('Deploy: gcloud run deploy orgiq-opa --image openpolicyagent/opa:latest --region', REGION)
print('Middleware integration: POST {OPA_URL}/v1/data/orgiq/authz/allow with {input: {principal, action, resource}}.')


## Done

Next: `10-identity-platform.ipynb` (issue JWTs that carry the `role` custom claim consumed by these CEL conditions and the OPA sidecar).
